<a href="https://colab.research.google.com/github/wuhao007/haowu999/blob/main/kaggle_haowu999.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance --quiet
import datetime
import numpy as np
import pandas as pd
import math
from sklearn.linear_model import LinearRegression
import yfinance as yf
import unittest
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
_AHR999_DAYS = 200

class Haowu999:
    """
    AHR999 定投/抄底指数重组版
    公式: ahr999 = (币价 / 200日均价) * (币价 / 拟合公允价)
    """

    def __init__(self, start_date='2009-01-03', coin='BTC-USD'):
        self.coin = coin
        self.start_date = pd.to_datetime(start_date)
        self.prices = None
        self.w = None
        self.b = None
        self.dates = None
        self.ydata = None
        self.predicted_ydata = None

    def load_data(self):
        """从 Yahoo Finance 获取最新日线数据"""
        print(f"Fetching data for {self.coin}...")
        df = yf.download(self.coin, start='2010-07-18', progress=False)
        if df.empty:
            raise ValueError("Failed to fetch data.")
        
        df = df.reset_index()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
            
        df = df[['Date', 'Close']].copy()
        df.columns = ['Date', 'Close']
        self.prices = df.dropna()
        self._fit_model()
        return self.prices

    def _fit_model(self):
        """执行幂律拟合: log10(Price) = w * log10(Days) + b"""
        df = self.prices.copy()
        df['Days'] = (df['Date'] - self.start_date).dt.days
        df = df[(df['Days'] > 0) & (df['Close'] > 1)].copy()
        
        self.dates = df['Date']
        x = np.log10(df['Days'].values).reshape(-1, 1)
        self.ydata = np.log10(df['Close'].values)
        
        model = LinearRegression().fit(x, self.ydata)
        self.w = model.coef_[0]
        self.b = model.intercept_
        self.predicted_ydata = model.predict(x)
        
        print(f"Model Fitted: w={self.w:.4f}, b={self.b:.4f}, R²={model.score(x, self.ydata):.4f}")

    def calculate_indicators(self, current_price=None):
        """计算当前的 ahr999 指数"""
        if self.prices is None:
            self.load_data()
            
        latest_data = self.prices.iloc[-1]
        p_now = current_price if current_price is not None else float(latest_data['Close'])
        date_now = latest_data['Date']
        
        past_199_sum = self.prices.iloc[-199:]['Close'].sum()
        ma200 = (past_199_sum + p_now) / 200
        
        days = (date_now - self.start_date).days
        fit_price = 10 ** (self.w * math.log10(days) + self.b)
        
        ahr999 = (p_now / ma200) * (p_now / fit_price)
        
        return {
            "date": date_now,
            "current_price": p_now,
            "ma200": ma200,
            "fit_price": fit_price,
            "ahr999": ahr999
        }

    def get_threshold(self, target_ahr999):
        """根据目标 ahr999 逆推币价"""
        latest_data = self.prices.iloc[-1]
        days = (latest_data['Date'] - self.start_date).days
        fit_price = 10 ** (self.w * math.log10(days) + self.b)
        sum199 = self.prices.iloc[-199:]['Close'].sum()
        
        a = 200
        b = - (target_ahr999 * fit_price)
        c = - (target_ahr999 * fit_price * sum199)
        
        delta = b**2 - 4*a*c
        if delta < 0: return np.nan
        return (-b + math.sqrt(delta)) / (2 * a)

In [ ]:
class TestHaowu999(unittest.TestCase):
    def test_logic(self):
        import pandas as pd
        # Mock data
        mock_data = pd.DataFrame({
            "Date": [pd.to_datetime("2020-01-01") + pd.Timedelta(days=i) for i in range(201)],
            "Close": [10000.0 + i * 10 for i in range(201)]
        })
        tester = Haowu999(start_date='2009-01-03')
        tester.prices = mock_data
        tester.w, tester.b = 5.6, -16.0
        
        p_45 = tester.get_threshold(0.45)
        res = tester.calculate_indicators(current_price=p_45)
        self.assertAlmostEqual(res['ahr999'], 0.45, places=5)
        print("Unit Test Passed: Mathematics is consistent.")

suite = unittest.TestLoader().loadTestsFromTestCase(TestHaowu999)
unittest.TextTestRunner(verbosity=1).run(suite)

In [ ]:
calc = Haowu999()
calc.load_data()
stats = calc.calculate_indicators()

print("\n" + "="*30)
print(f"BTC 分析结果 ({stats['date'].date()})")
print(f"当前价格: ${stats['current_price']:.2f}")
print(f"200日均线: ${stats['ma200']:.2f}")
print(f"拟合公允价: ${stats['fit_price']:.2f}")
print(f"AHR999 指数: {stats['ahr999']:.3f}")

p_045 = calc.get_threshold(0.45)
p_120 = calc.get_threshold(1.2)

print("-"*30)
print(f"抄底线 (0.45): ${p_045:.2f}")
print(f"定投线 (1.20): ${p_120:.2f}")
print("="*30)

In [ ]:
# --- 深度准确度审计 (Accuracy Audit) ---
from sklearn.metrics import mean_squared_error
import seaborn as sns

# 1. 计算对数价格残差 (Residuals = 实际 - 预测)
# 残差越接近 0，说明模型越准
residuals = calc.ydata - calc.predicted_ydata
rmse = np.sqrt(mean_squared_error(calc.ydata, calc.predicted_ydata))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 左图：残差随时间分布 (查看模型是否随时间发生漂移)
axes[0].plot(calc.dates, residuals, color="purple", alpha=0.5)
axes[0].axhline(y=0, color="black", linestyle="--")
axes[0].set_title(f"Model Residuals Over Time (RMSE: {rmse:.4f})")
axes[0].set_ylabel("Log Error")

# 右图：残差分布直方图 (查看误差是否符合正态分布)
sns.histplot(residuals, kde=True, ax=axes[1], color="green")
axes[1].set_title("Error Distribution (Bell Curve Test)")

plt.tight_layout()
plt.show()

print(f"拟合优度 (R²): {calc.w**2 / (calc.w**2 + np.var(residuals)):.4f}") # 简化版R2计算确认
print("结论：如果误差分布在 0 附近对称且没有明显的向上或向下倾斜，则模型在统计上是稳健的。")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 12))

axes[0].plot(calc.dates, 10**calc.ydata, label='Actual Price', alpha=0.6)
axes[0].plot(calc.dates, 10**calc.predicted_ydata, label='Regression Line', color='red')
axes[0].set_yscale('log')
axes[0].set_title("Bitcoin Price vs Regression")
axes[0].legend()

df_hist = calc.prices.copy()
df_hist['MA200'] = df_hist['Close'].rolling(200).mean()
df_hist['Days'] = (df_hist['Date'] - calc.start_date).dt.days
df_hist['Fit'] = 10 ** (calc.w * np.log10(df_hist['Days']) + calc.b)
df_hist['AHR999'] = (df_hist['Close'] / df_hist['MA200']) * (df_hist['Close'] / df_hist['Fit'])

axes[1].plot(df_hist['Date'], df_hist['AHR999'], label='AHR999')
axes[1].axhline(y=0.45, color='r', linestyle='--', label='0.45 (Bottom)')
axes[1].axhline(y=1.2, color='g', linestyle='--', label='1.2 (Invest)')
axes[1].set_yscale('log')
axes[1].set_title("Historical AHR999 Trend")
axes[1].legend()

plt.tight_layout()
plt.show()